## Parse & EDA

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
import gc
import psutil
import warnings
warnings.filterwarnings('ignore')

import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

In [ ]:
def get_ram_usage():
    """Get current RAM usage in GB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024 / 1024

print(f"Current RAM usage: {get_ram_usage():.2f} GB / 16 GB")

In [ ]:
# Get paths (notebook is in notebooks/ folder)
notebook_dir = Path.cwd()  
project_root = notebook_dir.parent 
data_raw = project_root / "data" / "raw_data"
data_processed = project_root / "data" / "processed_data"

print(f"Notebook location: {notebook_dir}")
print(f"Project root: {project_root}")
print(f"Raw data: {data_raw}")
print(f"Processed data: {data_processed}")
print()

# Verify paths exist
if not data_raw.exists():
    print(f"Raw data folder not found: {data_raw}")
else:
    print(f"Raw data folder found")

if not data_processed.exists():
    data_processed.mkdir(parents=True, exist_ok=True)
    print(f"Created processed data folder")

print()

# RAM monitoring function
def get_ram_usage():
    """Get current RAM usage in GB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024 / 1024

print(f"Current RAM usage: {get_ram_usage():.2f} GB")

### MedLayXPlain

In [ ]:
print("\n" + "=" * 70)
print("PARSING: MedLayXPlain Dataset")
print("=" * 70)
print()

# Check for parquet or csv files
medlay_parquet = list(medlay_dir.glob("*.parquet"))
medlay_csv = list(medlay_dir.glob("*.csv"))

if medlay_parquet:
    medlay_file = medlay_parquet[0]
    print(f"Found Parquet file: {medlay_file.name}")
elif medlay_csv:
    medlay_file = medlay_csv[0]
    print(f"Found CSV file: {medlay_file.name}")
else:
    print("✗ MedLayXPlain file not found!")
    medlay_file = None

if medlay_file:
    print(f"File size: {medlay_file.stat().st_size / 1024 / 1024 / 1024:.2f} GB")
    print(f"RAM before load: {get_ram_usage():.2f} GB")
    print()
    
    # Load with dtype optimization
    print("Loading MedLayXPlain...")
    try:
        if medlay_file.suffix == '.parquet':
            df_medlay = pd.read_parquet(medlay_file)
        else:
            # For CSV, specify dtypes to save memory
            df_medlay = pd.read_csv(
                medlay_file,
                dtype={
                    'image_id': 'str',
                    'modality': 'category',
                    'body_part': 'category',
                    'split': 'category'
                }
            )
        
        print(f"✓ Loaded successfully")
        print(f"RAM after load: {get_ram_usage():.2f} GB")
        print()
        
        # Dataset info
        print("Dataset Info:")
        print(f"  Shape: {df_medlay.shape}")
        print(f"  Columns: {list(df_medlay.columns)}")
        print()
        
        # Data types (memory efficient)
        print("Memory Usage by Column:")
        memory_usage = df_medlay.memory_usage(deep=True)
        total_memory = memory_usage.sum() / 1024 / 1024
        for col, mem in memory_usage.items():
            mem_mb = mem / 1024 / 1024
            print(f"  {col}: {mem_mb:.2f} MB")
        print(f"  TOTAL: {total_memory:.2f} MB")
        print()
        
        # Save metadata about the dataset
        metadata = {
            'samples': len(df_medlay),
            'columns': list(df_medlay.columns),
            'dtypes': df_medlay.dtypes.to_dict(),
            'memory_mb': total_memory
        }
        
        metadata_path = data_processed / "medlayxplain_metadata.json"
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2, default=str)
        print(f"✓ Saved metadata to: {metadata_path}")
        
    except Exception as e:
        print(f"✗ Error loading MedLayXPlain: {e}")
        df_medlay = None
else:
    df_medlay = None

print()

#### Loading the Dataset

In [ ]:
# Find MedLayXPlain files in medlayxplain subfolder
medlay_dir = data_raw / "medlayxplain"
print(f"Looking for MedLayXPlain in: {medlay_dir}")
print()

# List what's in the directory
if medlay_dir.exists():
    files = list(medlay_dir.glob("*"))
    print(f"Files found in {medlay_dir.name}:")
    for f in files:
        if f.is_file():
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  - {f.name} ({size_mb:.2f} MB)")
    print()
else:
    print(f"✗ Directory not found: {medlay_dir}")

# Find parquet or csv files
medlay_parquet = list(medlay_dir.glob("*.parquet"))
medlay_csv = list(medlay_dir.glob("*.csv"))

if medlay_parquet:
    medlay_file = medlay_parquet[0]
    file_format = "parquet"
elif medlay_csv:
    medlay_file = medlay_csv[0]
    file_format = "csv"
else:
    print("✗ No MedLayXPlain file found (expected .parquet or .csv)")
    medlay_file = None
    file_format = None

if medlay_file:
    print(f"Using file: {medlay_file.name}")
    print(f"Format: {file_format}")
    print(f"Size: {medlay_file.stat().st_size / 1024 / 1024 / 1024:.2f} GB")
    print()
    
    # Load with appropriate method
    print(f"Loading MedLayXPlain...")
    print(f"RAM before load: {get_ram_usage():.2f} GB")
    
    if file_format == "parquet":
        df_medlay = pd.read_parquet(medlay_file)
    else:  # CSV
        # For large CSV, read in chunks first to check size
        file_size_gb = medlay_file.stat().st_size / 1024 / 1024 / 1024
        if file_size_gb > 2:
            print("Large CSV detected - reading in chunks...")
            chunks = []
            for chunk in pd.read_csv(medlay_file, chunksize=50000):
                chunks.append(chunk)
                if len(chunks) % 10 == 0:
                    print(f"  Loaded {len(chunks) * 50000:,} rows...")
            df_medlay = pd.concat(chunks, ignore_index=True)
        else:
            df_medlay = pd.read_csv(medlay_file)
    
    print(f"✓ Loaded successfully")
    print(f"RAM after load: {get_ram_usage():.2f} GB")
    print()
    
else:
    print("Cannot proceed without MedLayXPlain file")
    df_medlay = None

#### Basic EDA

In [ ]:
if df_medlay is not None:
   
    # Shape
    print("Dataset Shape:")
    print(f"  Samples (rows): {len(df_medlay):,}")
    print(f"  Columns: {len(df_medlay.columns)}")
    print()
    
    # Column names and types
    print("Column Information:")
    print()
    for i, col in enumerate(df_medlay.columns, 1):
        dtype = df_medlay[col].dtype
        non_null = df_medlay[col].notna().sum()
        null_count = df_medlay[col].isna().sum()
        null_pct = (null_count / len(df_medlay)) * 100
        
        print(f"{i}. {col}")
        print(f" Type: {dtype},  Non-null: {non_null:,} ({100 - null_pct:.1f}%)")
        print()
    
    # Memory usage
    print("Memory Usage by Column:")
    memory_usage = df_medlay.memory_usage(deep=True)
    total_memory_mb = memory_usage.sum() / 1024 / 1024
    
    for col, mem in memory_usage.items():
        mem_mb = mem / 1024 / 1024
        print(f"  {col}: {mem_mb:.2f} MB")
    
    print(f"  TOTAL: {total_memory_mb:.2f} MB")
    print()

In [ ]:
if df_medlay is not None:
   
    print("First 3 Samples:")
    print()
    
    for idx in range(min(3, len(df_medlay))):
        print(f"SAMPLE {idx + 1}:")
        print("-" * 70)
        
        for col, value in df_medlay.iloc[idx].items():
            if pd.isna(value):
                print(f"  {col}: <NULL>")
            else:
                # Truncate long strings for readability
                val_str = str(value)
                if len(val_str) > 200:
                    print(f"  {col}:")
                    print(f"    {val_str[:200]}...")
                else:
                    print(f"  {col}: {val_str}")
        
        print()

In [ ]:
if df_medlay is not None:
    
    print("Data Types Summary:")
    dtype_counts = df_medlay.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  {dtype}: {count} columns")
    
    print()
    print("Unique Values by Column (for categorical/string columns):")
    print()
    
    # Show unique values for object/categorical columns
    object_cols = df_medlay.select_dtypes(include=['object', 'category']).columns
    
    for col in object_cols[:5]:  # First 5 object columns
        unique_count = df_medlay[col].nunique()
        print(f"  {col}: {unique_count} unique values")
        
        # Show top 5 values
        if unique_count <= 20:
            print(f"    All values:")
            for val, count in df_medlay[col].value_counts().items():
                pct = (count / len(df_medlay)) * 100
                print(f"      - {val}: {count:,} ({pct:.1f}%)")
        else:
            print(f"    Top 5 values:")
            for val, count in df_medlay[col].value_counts().head(5).items():
                pct = (count / len(df_medlay)) * 100
                print(f"      - {val}: {count:,} ({pct:.1f}%)")
        
        print()

#### Modality & Categorical Analysis 

In [ ]:
if df_medlay is not None:
    
    # Find text columns
    text_cols = [col for col in df_medlay.columns 
                 if any(kw in col.lower() for kw in ['text', 'report', 'caption', 'description', 'notes', 'findings'])]
    
    if text_cols:
        print(f"Found {len(text_cols)} text columns:")
        print()
        
        for col in text_cols:
            print(f"Column: {col}")
            
            # Length statistics
            lengths = df_medlay[col].fillna("").astype(str).str.len()
            
            print(f"  Length statistics:")
            print(f"    Mean: {lengths.mean():.0f} characters")
            print(f"    Median: {lengths.median():.0f} characters")
            print(f"    Std Dev: {lengths.std():.0f}")
            print(f"    Min: {lengths.min()}")
            print(f"    Max: {lengths.max()}")
            print()
            
            # Distribution
            print(f"  Non-empty: {(lengths > 0).sum():,}")
            print(f"  Empty/Null: {(lengths == 0).sum():,}")
            print()
            
            # Show sample
            non_null_values = df_medlay[col].dropna()
            if len(non_null_values) > 0:
                sample_text = str(non_null_values.iloc[0])  # Convert to string explicitly
                print(f"  Sample text:")
                print(f"    {sample_text[:300]}")
            else:
                print(f"  Sample text: <All values are NULL>")

    else:
        print("No obvious text columns found")

#### Summary

In [ ]:
if df_medlay is not None:
    print("\n" + "=" * 70)
    print("EDA SUMMARY")
    print("=" * 70)
    print()
    
    # Create summary
    eda_summary = {
        "timestamp": pd.Timestamp.now().isoformat(),
        "dataset": "MedLayXPlain",
        "samples": len(df_medlay),
        "columns": len(df_medlay.columns),
        "column_list": list(df_medlay.columns),
        "dtypes": {str(k): str(v) for k, v in df_medlay.dtypes.to_dict().items()},
        "memory_usage_mb": float(df_medlay.memory_usage(deep=True).sum() / 1024 / 1024),
        "missing_values": {col: int(count) for col, count in df_medlay.isnull().sum().items() if count > 0},
        "text_columns": text_cols if 'text_cols' in locals() else [],
        "categorical_columns": [col for col in df_medlay.columns if df_medlay[col].dtype == 'object' or df_medlay[col].dtype == 'category']
    }
    
    # Save to JSON
    report_path = data_processed / "medlayxplain_eda_summary.json"
    with open(report_path, 'w') as f:
        json.dump(eda_summary, f, indent=2, default=str)
    
    print(f"✓ EDA Summary saved to:")
    print(f"  {report_path}")
    print()
    
    # Print summary
    print("Summary:")
    print(f"  Samples: {eda_summary['samples']:,}")
    print(f"  Columns: {eda_summary['columns']}")
    print(f"  Memory: {eda_summary['memory_usage_mb']:.2f} MB")
    print(f"  Missing values: {len(eda_summary['missing_values'])} columns affected")
    print()
    
    print(f"Current RAM usage: {get_ram_usage():.2f} GB")

#### Word Counter (WILL CONTINUE LATER)

In [ ]:
if "df_medlay" in locals() and df_medlay is not None:

    report_col = (
        "radiology_report" if "radiology_report" in df_medlay.columns else None
    )

    if report_col:
        print(f"Targeting report text column: '{report_col}'")

        # Tokenize words using regex (words with length >= 3)
        raw_token_stream = []
        for text in df_medlay[report_col].dropna():
            words = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())
            raw_token_stream.extend(words)

        print(
            f"Total raw word tokens extracted across dataset: {len(raw_token_stream):,}\n"
        )

        # NTLK stopwords only
        print("--- ATTEMPT 1: NTLK Only ---")
        nltk_stopwords = set(stopwords.words("english"))

        pass1_words = [w for w in raw_token_stream if w not in nltk_stopwords]
        counter_pass1 = Counter(pass1_words)
        top_30_pass1 = counter_pass1.most_common(30)

        df_pass1 = pd.DataFrame(
            top_30_pass1, columns=["Term (NLTK Only)", "Count"]
        )
        print(df_pass1.to_string(index=False))
        print("\n" + "-" * 50 + "\n")

        # NLTK + Boilerplate
        print("--- ATTEMPT 2: NLTK + Boilerplate ---")

        # Only document structure / administrative metadata boilerplate.
        # NO biological or anatomical terms (e.g., 'bilateral', 'pleural', 'acute') are in this set.
        REPORT_BOILERPLATE = {
            "findings",
            "impression",
            "examination",
            "exam",
            "patient",
            "report",
            "image",
            "images",
            "view",
            "views",
            "pa",
            "radiograph",
            "chest",
            "noted",
            "seen",
            "show",
            "shows",
            "showing",
            "demonstrates",
            "demonstrated",
            "within",
            "limits",
            "normal",
            "compared",
            "comparison",
            "date",
        }

        combined_stopwords = nltk_stopwords.union(REPORT_BOILERPLATE)

        pass2_words = [
            w for w in raw_token_stream if w not in combined_stopwords
        ]
        counter_pass2 = Counter(pass2_words)
        top_30_pass2 = counter_pass2.most_common(30)

        df_pass2 = pd.DataFrame(
            top_30_pass2, columns=["Clinical Term (Boilerplate Removed)", "Count"]
        )
        print(df_pass2.to_string(index=False))

        # -------------------------------------------------------------
        # EXPORT RESULT TO PROCESSED_DATA
        # -------------------------------------------------------------
        extracted_terms = [t[0] for t in top_30_pass2]
        terms_out_path = processed_dir / "top_extracted_radiology_terms.json"
        with open(terms_out_path, "w") as f:
            json.dump(extracted_terms, f, indent=2)

        print()
        print(f"Saved top filtered clinical terms to: {terms_out_path}")

    else:
        print("No suitable radiology report text column found!")

else:
    print("'df_medlay' is not loaded. Please run the Step 1 loading cell first.")

### UMLS 2026AA

In [ ]:
print("\n" + "=" * 70)
print("UMLS 2026AA EXPLORATION")
print("=" * 70)
print()

# UMLS is in META subfolder
umls_meta_dir = data_raw / "umls" / "umls_2026AA_full" / "2026AA" /"META"

print(f"Looking for UMLS files in: {umls_meta_dir}")
print()

# Check if directory exists
if umls_meta_dir.exists():
    print(f"✓ UMLS META directory found")
    print()
    
    # List RRF files
    rrf_files = sorted(umls_meta_dir.glob("*.RRF"))
    ctl_files = sorted(umls_meta_dir.glob("*.ctl"))
    
    print(f"RRF Files ({len(rrf_files)}):")
    for f in rrf_files:
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  - {f.name}: {size_mb:.2f} MB")
    
    print()
    print(f"CTL Files ({len(ctl_files)}): (skipping these)")
    print()
    
else:
    print(f"✗ UMLS META directory not found: {umls_meta_dir}")
    print()
    print("Expected structure:")
    print("  D:\\Project\\Radiology_RAG\\data\\raw_data\\umls\\umls_2026AA_full\\META\\")

# Key RRF files we'll use
mrconso_file = umls_meta_dir / "MRCONSO.RRF"
mrdef_file = umls_meta_dir / "MRDEF.RRF"
mrsty_file = umls_meta_dir / "MRSTY.RRF"
mrrel_file = umls_meta_dir / "MRREL.RRF"

print("Key files:")
print(f"  MRCONSO exists: {mrconso_file.exists()}")
print(f"  MRDEF exists: {mrdef_file.exists()}")
print(f"  MRSTY exists: {mrsty_file.exists()}")
print(f"  MRREL exists: {mrrel_file.exists()}")
print()

#### MRCONSO.RRF STRUCTURE

MRCONSO.RRF Format:
- CUI = Concept Unique Identifier
- LAT = Language (ENG = English)
- SAB = Source/Vocabulary (MeSH, SNOMED, ICD-10, CHV, etc.)
- TTY = Term-Type (PT=Preferred, SY=Synonym, etc.)
- STR = String (the actual term)
- SUPRESS = Suppression flag (Y/N)

In [ ]:
if mrconso_file.exists():
     
    print("Reading first 100 records...")
    print()
       
    # Read first 100 rows
    df_sample = pd.read_csv(
        mrconso_file,
        sep='|',
        header=None,
        names=mrconso_cols,
        nrows=100,
        encoding='utf-8',
        on_bad_lines='skip',
        engine='c'
    )
    
    print(f"✓ Loaded {len(df_sample)} sample records")
    print()
    
    # Show a few records
    print("First 5 records (showing key columns only):")
    print()
    
    for idx in range(min(5, len(df_sample))):
        row = df_sample.iloc[idx]
        print(f"Record {idx + 1}:")
        print(f"  CUI: {row['CUI']}")
        print(f"  Term: {row['STR']}")
        print(f"  Source (SAB): {row['SAB']}")
        print(f"  Term Type (TTY): {row['TTY']}")
        print(f"  Language: {row['LAT']}")
        print(f"  Preferred: {row['ISPREF']}")
        print()
    
    print("Column data types in sample:")
    print(df_sample.dtypes)

else:
    print("✗ MRCONSO.RRF not found, skipping sample")

print()

In [ ]:
if mrconso_file.exists():


    # --- Step 1: discover the actual columns from MRFILES.RRF instead of hardcoding ---
    mrfiles_path = mrconso_file.parent / "MRFILES.RRF"
    mrconso_cols = None

    if mrfiles_path.exists():
        print("Discovering MRCONSO columns from MRFILES.RRF...")
        with open(mrfiles_path, encoding='utf-8') as f:
            for line in f:
                fields = line.rstrip('\n').split('|')
                # MRFILES.RRF format: FIL|DES|FMT|CLS|RWS|BTS
                if fields[0] == 'MRCONSO.RRF':
                    mrconso_cols = fields[2].split(',')
                    break

    if mrconso_cols:
        print(f"✓ Found {len(mrconso_cols)} columns: {mrconso_cols}")
    else:
        print("✗ Could not find MRCONSO.RRF entry in MRFILES.RRF, falling back to raw inspection")
    print()

    # --- Step 2: peek at raw lines to confirm actual field count in the data itself ---
    print("Peeking at first 5 raw lines to verify field count...")
    with open(mrconso_file, encoding='utf-8') as f:
        raw_lines = [next(f) for _ in range(5)]

    field_counts = set()
    for line in raw_lines:
        fields = line.rstrip('\n').split('|')
        field_counts.add(len(fields))
    print(f"Field counts seen in first 5 lines: {field_counts}")

    # If MRFILES.RRF wasn't available, derive a generic column list from the raw field count
    if not mrconso_cols:
        n_fields = max(field_counts)
        mrconso_cols = [f"COL{i}" for i in range(n_fields)]
        print(f"Using generic column names: {mrconso_cols}")
    print()

    # --- Step 3: read first 100 rows using the discovered columns ---
    print("Reading first 100 records...")
    print()

    df_sample = pd.read_csv(
        mrconso_file,
        sep='|',
        header=None,
        names=mrconso_cols,
        index_col=False,   # avoids misalignment from the trailing pipe
        nrows=100,
        encoding='utf-8',
        on_bad_lines='skip',
        engine='c'
    )

    print(f"✓ Loaded {len(df_sample)} sample records")
    print()

    print("First 5 records (raw, all columns):")
    print()
    for idx in range(min(5, len(df_sample))):
        row = df_sample.iloc[idx]
        print(f"Record {idx + 1}:")
        for col in mrconso_cols:
            print(f"  {col}: {row[col]}")
        print()

    print("Column data types in sample:")
    print(df_sample.dtypes)

else:
    print("MRCONSO.RRF not found, skipping sample")

print()

In [ ]:
if mrconso_file.exists():
    print(f"RAM before: {get_ram_usage():.2f} GB")

    total_rows = 0
    unique_cuis = set()
    language_dist = {}
    source_dist = {}
    term_type_dist = {}
    chunk_num = 0

    for chunk in pd.read_csv(
        mrconso_file,
        sep='|',
        header=None,
        names=mrconso_cols,
        index_col=False,      # <-- critical fix: prevents the trailing '|' from
                               #     being misread as an index column, which was
                               #     shifting every column left by one
        encoding='utf-8',
        on_bad_lines='skip',
        engine='c',
        chunksize=1_000_000,
        dtype={
            'CUI': 'str',
            'LAT': 'category',
            'SAB': 'category',
            'TTY': 'category',
            'STR': 'str'
        }
    ):
        chunk_num += 1
        total_rows += len(chunk)
        
        # Collect unique CUIs
        unique_cuis.update(chunk['CUI'].unique().tolist())
        
        # Count languages
        for lang, count in chunk['LAT'].value_counts().items():
            language_dist[lang] = language_dist.get(lang, 0) + count
        
        # Count sources
        for source, count in chunk['SAB'].value_counts().items():
            source_dist[source] = source_dist.get(source, 0) + count
        
        # Count term types
        for ttype, count in chunk['TTY'].value_counts().items():
            term_type_dist[ttype] = term_type_dist.get(ttype, 0) + count
        
        # Progress
        if chunk_num % 5 == 0:
            print(f"  Chunk {chunk_num}: {total_rows:,} rows | "
                  f"{len(unique_cuis):,} unique CUIs | "
                  f"RAM: {get_ram_usage():.2f} GB")
        
        gc.collect()  # Force garbage collection
    
    print()
    print("✓ Processing complete")
    print()
    
    # Print results
    print("STATISTICS:")
    print(f"  Total term records: {total_rows:,}")
    print(f"  Unique concepts (CUIs): {len(unique_cuis):,}")
    print(f"  Average terms per concept: {total_rows / len(unique_cuis):.1f}")
    print()
    
    print("Languages (LAT):")
    for lang in sorted(language_dist.keys(), key=lambda x: language_dist[x], reverse=True):
        count = language_dist[lang]
        pct = (count / total_rows) * 100
        print(f"  {lang}: {count:,} ({pct:.1f}%)")
    
    print()
    print("Top 15 Vocabulary Sources (SAB):")
    sorted_sources = sorted(source_dist.items(), key=lambda x: x[1], reverse=True)
    for source, count in sorted_sources[:15]:
        pct = (count / total_rows) * 100
        print(f"  {source}: {count:,} ({pct:.1f}%)")
    
    print()
    print("Term Types (TTY):")
    sorted_ttypes = sorted(term_type_dist.items(), key=lambda x: x[1], reverse=True)
    for ttype, count in sorted_ttypes:
        pct = (count / total_rows) * 100
        print(f"  {ttype}: {count:,} ({pct:.1f}%)")
    
    print()
    print(f"RAM after: {get_ram_usage():.2f} GB")

else:
    print("✗ MRCONSO.RRF not found")

print()


In [ ]:
if mrconso_file.exists():
    print("Searching for concepts containing chest-related keywords...")
    print()
    
    chest_keywords = [
        'lung', 'chest', 'thorax', 'thoracic', 'heart', 'cardiac', 'cardio',
        'pneumonia', 'pneumothorax', 'bronch', 'trachea', 'mediastin',
        'pleura', 'rib', 'intercost', 'diaphragm', 'pulmonary', 'respir',
        'clavicle', 'scapula'
    ]
    
    chest_cuis = set()
    chest_terms = []
    
    chunk_num = 0
    total_scanned = 0
    
    print("Scanning MRCONSO for chest concepts...")
    print()
    
    for chunk in pd.read_csv(
        mrconso_file,
        sep='|',
        header=None,
        names=mrconso_cols,
        encoding='utf-8',
        on_bad_lines='skip',
        engine='c',
        chunksize=100000,
        dtype={'CUI': 'str', 'SAB': 'category', 'STR': 'str'}
    ):
        chunk_num += 1
        total_scanned += len(chunk)
        
        # Find rows with chest keywords
        mask = chunk['STR'].fillna("").str.lower().str.contains(
            '|'.join(chest_keywords), regex=True, na=False
        )
        
        # Save matching CUIs and terms
        chest_chunk = chunk[mask]
        chest_cuis.update(chest_chunk['CUI'].unique().tolist())
        
        for idx, row in chest_chunk.iterrows():
            chest_terms.append({
                'CUI': row['CUI'],
                'Term': row['STR'],
                'Source': row['SAB']
            })
        
        if chunk_num % 5 == 0:
            print(f"  Chunk {chunk_num}: Scanned {total_scanned:,} | Found {len(chest_cuis):,} chest CUIs")
        
        gc.collect()
    
    print()
    print("✓ Search complete")
    print()
    
    print("RESULTS:")
    print(f"  Total UMLS records scanned: {total_scanned:,}")
    print(f"  Unique chest-related CUIs found: {len(chest_cuis):,}")
    print(f"  Chest term records found: {len(chest_terms):,}")
    print()
    
    # Sample chest terms
    print("Sample chest-related terms (first 20):")
    chest_df = pd.DataFrame(chest_terms[:20])
    for idx, row in chest_df.iterrows():
        print(f"  - {row['Term']} (CUI: {row['CUI']}, Source: {row['Source']})")
    
    print()

else:
    print("MRCONSO.RRF not found")

print()

In [ ]:
#CHECK OTHER FILES
# MRDEF - Definitions
if mrdef_file.exists():
    print(f"MRDEF.RRF (Definitions):")
    print(f"  Size: {mrdef_file.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"  Format: CUI|AUI|ATUI|SATUI|SAB|DEF|RELA|FTCF|CXN")
    print(f"  Purpose: Definition text for concepts")
    print()
else:
    print("✗ MRDEF.RRF not found")

# MRSTY - Semantic Types
if mrsty_file.exists():
    print(f"MRSTY.RRF (Semantic Types):")
    print(f"  Size: {mrsty_file.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"  Format: CUI|TUI|STN|STY|ATUI|CVF")
    print(f"  Purpose: Semantic type (Anatomy, Finding, Disease, etc.)")
    print()
else:
    print("✗ MRSTY.RRF not found")

# MRREL - Relationships
if mrrel_file.exists():
    print(f"MRREL.RRF (Relationships):")
    print(f"  Size: {mrrel_file.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"  Format: CUI1|AUI1|STYPE1|REL|CUI2|AUI2|STYPE2|RELA|RUI|SRUI|SAB|SL|RG|SUPPRESS|CVF")
    print(f"  Purpose: Relationships between concepts")
    print()
else:
    print("✗ MRREL.RRF not found")

print()

#### MRDEF.RRF STRUCTURE

In [ ]:
mrdef_file = umls_meta_dir / "MRDEF.RRF"

if mrdef_file.exists():
    mrfiles_path = mrdef_file.parent / "MRFILES.RRF"
    mrdef_cols = None

    if mrfiles_path.exists():
        print("Discovering MRDEF columns from MRFILES.RRF...")
        with open(mrfiles_path, encoding='utf-8') as f:
            for line in f:
                fields = line.rstrip('\n').split('|')

                if fields[0] == 'MRDEF.RRF':
                    mrdef_cols = fields[2].split(',')
                    break

    if mrdef_cols:
        print(f"✓ Found {len(mrdef_cols)} columns: {mrdef_cols}")
    else:
        print("✗ Could not find MRDEF.RRF entry in MRFILES.RRF, falling back to raw inspection")
    print()

    # --- Step 2: peek at raw lines to confirm actual field count in the data itself ---
    print("Peeking at first 5 raw lines to verify field count...")
    with open(mrdef_file, encoding='utf-8') as f:
        raw_lines = [next(f) for _ in range(5)]

    field_counts = set()
    for line in raw_lines:
        fields = line.rstrip('\n').split('|')
        field_counts.add(len(fields))
    print(f"Field counts seen in first 5 lines: {field_counts}")

    # If MRFILES.RRF wasn't available, derive a generic column list from the raw field count
    if not mrdef_cols:
        n_fields = max(field_counts)
        mrdef_cols = [f"COL{i}" for i in range(n_fields)]
        print(f"Using generic column names: {mrdef_cols}")
    print()

    # --- Step 3: read first 100 rows using the discovered columns ---
    print("Reading first 100 records...")
    print()

    df_sample = pd.read_csv(
        mrdef_file,
        sep='|',
        header=None,
        names=mrdef_cols,
        index_col=False,   # avoids misalignment from the trailing pipe
        nrows=100,
        encoding='utf-8',
        on_bad_lines='skip',
        engine='c'
    )

    print(f"✓ Loaded {len(df_sample)} sample records")
    print()

    print("First 5 records (raw, all columns):")
    print()
    for idx in range(min(5, len(df_sample))):
        row = df_sample.iloc[idx]
        print(f"Record {idx + 1}:")
        for col in mrdef_cols:
            print(f"  {col}: {row[col]}")
        print()

    print("Column data types in sample:")
    print(df_sample.dtypes)

else:
    print("✗ MRDEF.RRF not found, skipping sample")

print()

#### MRSTY.RRF STRUCTURE

In [ ]:
# mrsty_file already defined as: mrsty_file = umls_meta_dir / "MRSTY.RRF"

if mrsty_file.exists():
    # --- Step 1: discover the actual columns from MRFILES.RRF instead of hardcoding ---
    mrfiles_path = mrsty_file.parent / "MRFILES.RRF"
    mrsty_cols = None

    if mrfiles_path.exists():
        print("Discovering MRSTY columns from MRFILES.RRF...")
        with open(mrfiles_path, encoding='utf-8') as f:
            for line in f:
                fields = line.rstrip('\n').split('|')
                # MRFILES.RRF format: FIL|DES|FMT|CLS|RWS|BTS
                if fields[0] == 'MRSTY.RRF':
                    mrsty_cols = fields[2].split(',')
                    break

    if mrsty_cols:
        print(f"✓ Found {len(mrsty_cols)} columns: {mrsty_cols}")
    else:
        print("✗ Could not find MRSTY.RRF entry in MRFILES.RRF, falling back to raw inspection")
    print()

    # --- Step 2: peek at raw lines to confirm actual field count in the data itself ---
    print("Peeking at first 5 raw lines to verify field count...")
    with open(mrsty_file, encoding='utf-8') as f:
        raw_lines = [next(f) for _ in range(5)]

    field_counts = set()
    for line in raw_lines:
        fields = line.rstrip('\n').split('|')
        field_counts.add(len(fields))
    print(f"Field counts seen in first 5 lines: {field_counts}")

    # If MRFILES.RRF wasn't available, derive a generic column list from the raw field count
    if not mrsty_cols:
        n_fields = max(field_counts)
        mrsty_cols = [f"COL{i}" for i in range(n_fields)]
        print(f"Using generic column names: {mrsty_cols}")
    print()

    # --- Step 3: read first 100 rows using the discovered columns ---
    print("Reading first 100 records...")
    print()

    df_sample = pd.read_csv(
        mrsty_file,
        sep='|',
        header=None,
        names=mrsty_cols,
        index_col=False,   # avoids misalignment from the trailing pipe
        nrows=100,
        encoding='utf-8',
        on_bad_lines='skip',
        engine='c'
    )

    print(f"✓ Loaded {len(df_sample)} sample records")
    print()

    print("First 5 records (raw, all columns):")
    print()
    for idx in range(min(5, len(df_sample))):
        row = df_sample.iloc[idx]
        print(f"Record {idx + 1}:")
        for col in mrsty_cols:
            print(f"  {col}: {row[col]}")
        print()

    print("Column data types in sample:")
    print(df_sample.dtypes)

else:
    print("✗ MRSTY.RRF not found, skipping sample")

print()

#### Filtering UMLS

In [ ]:
#Filter ONLY LAT = ENG

if mrconso_file.exists():
    print(f"RAM before: {get_ram_usage():.2f} GB")

    # --- Setup output path ---
    output_dir = Path(r"D:\Project\Radiology_RAG\data\processed_data\filtered_data")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "MRCONSO.RRF"

    # If a previous run left a file behind, remove it so we don't append to stale data
    if output_file.exists():
        output_file.unlink()

    total_rows_read = 0
    total_rows_kept = 0
    chunk_num = 0
    first_chunk = True

    for chunk in pd.read_csv(
        mrconso_file,
        sep='|',
        header=None,
        names=mrconso_cols,
        index_col=False,       # avoid the trailing '|' shifting columns
        encoding='utf-8',
        on_bad_lines='skip',
        engine='c',
        chunksize=1_000_000,
        dtype={
            'CUI': 'str',
            'LAT': 'category',
            'SAB': 'category',
            'TTY': 'category',
            'STR': 'str'
        }
    ):
        chunk_num += 1
        total_rows_read += len(chunk)

        # --- Filter: English only ---
        filtered = chunk[chunk['LAT'] == 'ENG']
        total_rows_kept += len(filtered)

        # --- Write out in RRF format (pipe-delimited, trailing '|', no header) ---
        # Adding an empty final column reproduces the trailing '|' that real
        # RRF files end each line with, so downstream tools parse it the same way.
        filtered = filtered.copy()
        filtered[''] = ''

        filtered.to_csv(
            output_file,
            sep='|',
            header=False,
            index=False,
            mode='a' if not first_chunk else 'w',
            encoding='utf-8'
        )
        first_chunk = False

        if chunk_num % 5 == 0:
            print(f"  Chunk {chunk_num}: {total_rows_read:,} rows read | "
                  f"{total_rows_kept:,} ENG rows kept | "
                  f"RAM: {get_ram_usage():.2f} GB")

        gc.collect()

    print()
    print("✓ Filtering complete")
    print()
    print(f"  Total rows read:    {total_rows_read:,}")
    print(f"  English rows kept:  {total_rows_kept:,} ({total_rows_kept/total_rows_read*100:.1f}%)")
    print(f"  Output written to:  {output_file}")
    print()
    print(f"RAM after: {get_ram_usage():.2f} GB")

else:
    print("✗ MRCONSO.RRF not found")

print()

In [ ]:
from collections import Counter
from pathlib import Path
import psutil

ROOT = Path.cwd().parent
FILT = ROOT / "data" / "processed_data" / "filtered_data"
SRC = FILT / "MRCONSO_ENG.RRF"
DST = FILT / "MRCONSO_ENG_N.RRF"

def ram():
    return f"{psutil.Process().memory_info().rss / 1e9:.2f} GB"

# Peek first: confirm the format before filtering
with open(SRC, encoding="utf-8", newline="") as f:
    for _ in range(2):
        print(repr(f.readline()[:200]))

SUP_IDX, LAT_IDX = 16, 1     # 0-based positions in MRCONSO
sup_counts, lat_counts = Counter(), Counter()
total = kept = bad = 0

with open(SRC, encoding="utf-8", newline="") as fin, \
     open(DST, "w", encoding="utf-8", newline="") as fout:
    for line in fin:
        total += 1
        parts = line.split("|")
        if len(parts) <= SUP_IDX:        # malformed line, count and skip
            bad += 1
            continue
        sup_counts[parts[SUP_IDX]] += 1
        lat_counts[parts[LAT_IDX]] += 1
        if parts[SUP_IDX] == "N":
            fout.write(line)
            kept += 1
        if total % 2_000_000 == 0:
            print(f"{total:,} lines | kept {kept:,} | RAM {ram()}")

print(f"\nTotal lines : {total:,}")
print(f"Kept (N)    : {kept:,}")
print(f"Malformed   : {bad:,}")
print("SUPPRESS values:", dict(sup_counts))
print("LAT values     :", dict(lat_counts))
print("Output size (GB):", round(DST.stat().st_size / 1e9, 2))

### MeSH desc2026 XML

In [ ]:
mesh_file = Path(r"D:\Project\Radiology_RAG\data\raw_data\mesh\desc2026\desc2026.xml")

if mesh_file.exists():
    # PART 1: BASIC EDA — file-level structure
    file_size_gb = mesh_file.stat().st_size / (1024 ** 3)
    print(f"File: {mesh_file.name}")
    print(f"Size: {file_size_gb:.3f} GB")
    print()

    # Peek at the root tag and top-level structure without loading the whole tree
    print("Peeking at root element...")
    context = ET.iterparse(mesh_file, events=('start',))
    _, root_elem = next(context)
    print(f"  Root tag: {root_elem.tag}")
    print(f"  Root attributes: {root_elem.attrib}")
    del context
    print()

    # Count DescriptorRecord elements + basic tag inventory from the first record
    print("Scanning first DescriptorRecord for structure...")
    context = ET.iterparse(mesh_file, events=('end',))
    first_record_tags = None
    for event, elem in context:
        if elem.tag == 'DescriptorRecord' and first_record_tags is None:
            first_record_tags = [child.tag for child in elem]
            break
    del context
    gc.collect()

    print("  Top-level child tags of a DescriptorRecord:")
    for tag in first_record_tags:
        print(f"    - {tag}")
    print()

In [ ]:
if mesh_file.exists():


    def walk(elem, path, path_counter, path_attribs, path_has_text, sample_records=None, max_depth=25):
        if len(path) > max_depth:
            return
        current_path = "/".join(path)
        path_counter[current_path] += 1

        # Track which attributes appear on elements at this path
        if elem.attrib:
            path_attribs[current_path].update(elem.attrib.keys())

        # Track whether this path ever carries direct text content
        if elem.text and elem.text.strip():
            path_has_text[current_path] = True

        for child in elem:
            walk(child, path + [child.tag], path_counter, path_attribs, path_has_text, sample_records, max_depth)

    path_counter = Counter()        # how many times each relative path occurs (within one record)
    path_attribs = defaultdict(set) # attributes seen at each path
    path_has_text = {}              # whether a path ever has text content

    # --- Discover structure across the first N records, not just 1 ---
    # (some optional tags like PharmacologicalActionList won't appear in record #1)
    N_SAMPLE_RECORDS = 200
    records_scanned = 0

    context = ET.iterparse(mesh_file, events=('end',))
    for event, elem in context:
        if elem.tag != 'DescriptorRecord':
            continue

        records_scanned += 1
        walk(elem, ['DescriptorRecord'], path_counter, path_attribs, path_has_text)

        elem.clear()

        if records_scanned >= N_SAMPLE_RECORDS:
            break

    del context
    gc.collect()

    print(f"Scanned {records_scanned} DescriptorRecord(s) to build schema map")
    print()

    # Present the discovered structure
    print("-" * 70)
    print("ALL PATHS FOUND (relative to DescriptorRecord)")
    print("-" * 70)
    print()

    discovered_paths = sorted(path_counter.keys())
    for p in discovered_paths:
        occurs_pct = path_counter[p] / records_scanned * 100
        attribs = sorted(path_attribs.get(p, []))
        has_text = path_has_text.get(p, False)

        line = f"  {p}"
        line += f"  [seen in {occurs_pct:.0f}% of sampled records]"
        if has_text:
            line += "  [has text]"
        if attribs:
            line += f"  [attribs: {', '.join(attribs)}]"
        print(line)

    print()
    print(f"Total unique paths discovered: {len(discovered_paths)}")
    print()

    # Save into a plain list for the next script to iterate over / reference
    mesh_discovered_paths = discovered_paths
    mesh_path_attribs = dict(path_attribs)
    mesh_path_has_text = dict(path_has_text)

    print("Saved into variables: mesh_discovered_paths, mesh_path_attribs, mesh_path_has_text")

else:
    print(f"✗ File not found: {mesh_file}")

print()

In [ ]:
import math
MAX_DISTINCT_ATTRIB_VALUES = 50   # safety cap so a high-cardinality attribute can't blow up RAM

class RunningStats:
    """O(1) memory running stats — no raw values stored."""
    __slots__ = ('n', 'total', 'sumsq', 'min', 'max')
    def __init__(self):
        self.n = 0
        self.total = 0
        self.sumsq = 0
        self.min = None
        self.max = None
    def update(self, value):
        self.n += 1
        self.total += value
        self.sumsq += value * value
        if self.min is None or value < self.min:
            self.min = value
        if self.max is None or value > self.max:
            self.max = value
    @property
    def mean(self):
        return self.total / self.n if self.n else 0.0
    @property
    def std(self):
        if self.n < 2:
            return 0.0
        var = (self.sumsq / self.n) - (self.mean ** 2)
        return math.sqrt(max(var, 0.0))


if mesh_file.exists() and mesh_discovered_paths:

    print("=" * 70)
    print("MESH XML — FULL COUNTING & DISTRIBUTIONS (schema-driven)")
    print("=" * 70)
    print()

    # --- Per-path structures, all O(1)-ish memory, sized by number of paths (small) ---
    occurrences_per_record = {p: RunningStats() for p in mesh_discovered_paths}
    records_containing_path = Counter()          # how many records have >=1 occurrence of this path
    attrib_value_counts = {                        # path -> attrib_name -> Counter(value)
        p: {a: Counter() for a in mesh_path_attribs.get(p, [])}
        for p in mesh_discovered_paths if mesh_path_attribs.get(p)
    }
    attrib_high_cardinality = defaultdict(set)      # path -> set of attribs we stopped tracking
    text_length_stats = {
        p: RunningStats() for p in mesh_discovered_paths if mesh_path_has_text.get(p)
    }

    total_records = 0

    def walk(elem, path, local_counts):
        current_path = "/".join(path)
        local_counts[current_path] += 1

        # --- attribute value distributions (updated directly, per-element occurrence) ---
        if elem.attrib and current_path in attrib_value_counts:
            for k, v in elem.attrib.items():
                counters = attrib_value_counts[current_path]
                if k not in counters:
                    continue
                if k in attrib_high_cardinality[current_path]:
                    continue
                counters[k][v] += 1
                if len(counters[k]) > MAX_DISTINCT_ATTRIB_VALUES:
                    attrib_high_cardinality[current_path].add(k)
                    counters[k] = Counter()  # drop — too high-cardinality to be useful

        # --- text length stats (updated directly, per-element occurrence) ---
        if current_path in text_length_stats and elem.text and elem.text.strip():
            text_length_stats[current_path].update(len(elem.text.strip()))

        for child in elem:
            walk(child, path + [child.tag], local_counts)

    print("Streaming through file...")
    print()

    context = ET.iterparse(mesh_file, events=('end',))
    for event, elem in context:
        if elem.tag != 'DescriptorRecord':
            continue

        total_records += 1
        local_counts = Counter()
        walk(elem, ['DescriptorRecord'], local_counts)

        # Roll this record's per-path counts into the global running stats
        for p in mesh_discovered_paths:
            count = local_counts.get(p, 0)
            occurrences_per_record[p].update(count)
            if count > 0:
                records_containing_path[p] += 1

        elem.clear()  # free memory — critical for large files

        if total_records % 5000 == 0:
            print(f"  Processed {total_records:,} records...")

    del context
    gc.collect()

    print()
    print(f"Finished parsing {total_records:,} DescriptorRecords")
    print()

    # REPORT
    print("PATH OCCURRENCE SUMMARY")
    print()

    for p in sorted(mesh_discovered_paths):
        stats = occurrences_per_record[p]
        present_count = records_containing_path[p]
        present_pct = present_count / total_records * 100 if total_records else 0

        print(f"{p}")
        print(f"  Present in: {present_count:,}/{total_records:,} records ({present_pct:.1f}%)")
        print(f"  Count per record — min={stats.min}, max={stats.max}, "
              f"mean={stats.mean:.2f}, std={stats.std:.2f}")

        # Attribute value distributions, if this path has attributes
        if p in attrib_value_counts:
            for attrib_name, counter in attrib_value_counts[p].items():
                if attrib_name in attrib_high_cardinality[p]:
                    print(f"  Attribute '{attrib_name}': high-cardinality, skipped")
                    continue
                if not counter:
                    continue
                top_vals = counter.most_common(10)
                total_attrib = sum(counter.values())
                print(f"  Attribute '{attrib_name}' distribution (top 10):")
                for val, cnt in top_vals:
                    pct = cnt / total_attrib * 100
                    print(f"    {val}: {cnt:,} ({pct:.1f}%)")

        # Text length stats, if this path carries text
        if p in text_length_stats and text_length_stats[p].n > 0:
            ts = text_length_stats[p]
            print(f"  Text length (chars) — min={ts.min}, max={ts.max}, "
                  f"mean={ts.mean:.1f}, std={ts.std:.1f}, n={ts.n:,}")

else:
    if not mesh_file.exists():
        print(f"✗ File not found: {mesh_file}")
    else:
        print("✗ mesh_discovered_paths is empty — run the discovery script first")

print()

In [ ]:
N_SAMPLES = 5

if mesh_file.exists():

    print("=" * 70)
    print(f"MESH XML — {N_SAMPLES} SAMPLE RECORDS")
    print("=" * 70)
    print()

    def extract_text(elem, relative_path):
        """Given a DescriptorRecord elem and a path like 'ConceptList/Concept/ConceptName/String',
        return a list of text values found at that path (there may be several, e.g. multiple Concepts)."""
        results = []
        for found in elem.findall(relative_path):
            if found.text and found.text.strip():
                results.append(found.text.strip())
        return results

    samples_collected = 0
    context = ET.iterparse(mesh_file, events=('end',))

    for event, elem in context:
        if elem.tag != 'DescriptorRecord':
            continue

        samples_collected += 1

        print("-" * 70)
        print(f"SAMPLE RECORD {samples_collected}")
        print("-" * 70)

        # --- Record-level attributes ---
        print(f"DescriptorRecord attributes: {elem.attrib}")
        print()

        # --- DescriptorUI / DescriptorName (core identity) ---
        ui = extract_text(elem, 'DescriptorUI')
        name = extract_text(elem, 'DescriptorName/String')
        print(f"DescriptorUI:   {ui[0] if ui else '(none)'}")
        print(f"DescriptorName: {name[0] if name else '(none)'}")
        print()

        # --- Tree numbers ---
        tree_numbers = extract_text(elem, 'TreeNumberList/TreeNumber')
        print(f"TreeNumbers ({len(tree_numbers)}): {tree_numbers}")
        print()

        # --- Concepts ---
        concepts = elem.findall('ConceptList/Concept')
        print(f"Concepts ({len(concepts)}):")
        for c in concepts:
            c_name = extract_text(c, 'ConceptName/String')
            c_pref = c.attrib.get('PreferredConceptYN', '?')
            print(f"  - {c_name[0] if c_name else '(unnamed)'}  [PreferredConceptYN={c_pref}]")

            # Terms under this concept
            terms = c.findall('TermList/Term')
            for t in terms:
                t_str = extract_text(t, 'String')
                t_pref = t.attrib.get('ConceptPreferredTermYN', '?')
                print(f"      Term: {t_str[0] if t_str else '(none)'}  [ConceptPreferredTermYN={t_pref}]")

            # Scope note under this concept, if present
            scope_note = extract_text(c, 'ScopeNote')
            if scope_note:
                note_preview = scope_note[0][:150] + ("..." if len(scope_note[0]) > 150 else "")
                print(f"      ScopeNote: {note_preview}")
        print()

        # --- Pharmacological actions, if present ---
        pharm_actions = elem.findall('PharmacologicalActionList/PharmacologicalAction')
        if pharm_actions:
            print(f"PharmacologicalActions ({len(pharm_actions)}):")
            for pa in pharm_actions:
                pa_name = extract_text(pa, 'DescriptorReferredTo/DescriptorName/String')
                print(f"  - {pa_name[0] if pa_name else '(unnamed)'}")
            print()

        # --- Date established ---
        year = extract_text(elem, 'DateEstablished/Year')
        if year:
            print(f"DateEstablished (Year): {year[0]}")
            print()

        elem.clear()  # free memory

        if samples_collected >= N_SAMPLES:
            break

    del context
    gc.collect()

    print("=" * 70)
    print(f"✓ Retrieved {samples_collected} sample record(s)")

else:
    print(f"✗ File not found: {mesh_file}")

print()

In [ ]:
import xml.etree.ElementTree as ET
import gc

# Assumes these already exist from the discovery script:
# mesh_discovered_paths, mesh_path_attribs, mesh_path_has_text, mesh_file

N_SAMPLES = 5

if mesh_file.exists() and mesh_discovered_paths:

    print("=" * 70)
    print(f"MESH XML — {N_SAMPLES} SAMPLE RECORDS (schema-driven)")
    print("=" * 70)
    print()

    # Convert discovered paths (which include the "DescriptorRecord/" root)
    # into relative paths usable with elem.findall() from a DescriptorRecord element,
    # and sort by depth so output prints in a roughly hierarchical order.
    relative_paths = []
    for p in mesh_discovered_paths:
        parts = p.split("/")
        if parts[0] == "DescriptorRecord":
            parts = parts[1:]
        if not parts:
            continue  # this *is* DescriptorRecord itself, nothing to traverse
        relative_paths.append((len(parts), "/".join(parts), p))  # (depth, relative_path, full_path)

    relative_paths.sort()  # shallow paths first

    samples_collected = 0
    context = ET.iterparse(mesh_file, events=('end',))

    for event, elem in context:
        if elem.tag != 'DescriptorRecord':
            continue

        samples_collected += 1

        print("-" * 70)
        print(f"SAMPLE RECORD {samples_collected}")
        print("-" * 70)
        print(f"DescriptorRecord attributes: {elem.attrib}")
        print()

        for depth, rel_path, full_path in relative_paths:
            found_elements = elem.findall(rel_path)
            if not found_elements:
                continue  # this optional path just isn't present in this record

            indent = "  " * (depth - 1)
            print(f"{indent}{full_path}  ({len(found_elements)} found)")

            for fe in found_elements:
                line = f"{indent}  ->"

                has_text = mesh_path_has_text.get(full_path, False)
                if has_text and fe.text and fe.text.strip():
                    text_val = fe.text.strip()
                    preview = text_val[:150] + ("..." if len(text_val) > 150 else "")
                    line += f" text='{preview}'"

                attribs_of_interest = mesh_path_attribs.get(full_path)
                if attribs_of_interest and fe.attrib:
                    shown_attribs = {k: v for k, v in fe.attrib.items() if k in attribs_of_interest}
                    if shown_attribs:
                        line += f" attribs={shown_attribs}"

                if line != f"{indent}  ->":
                    print(line)

        print()

        elem.clear()  # free memory

        if samples_collected >= N_SAMPLES:
            break

    del context
    gc.collect()

    print("=" * 70)
    print(f"✓ Retrieved {samples_collected} sample record(s)")

elif not mesh_file.exists():
    print(f"✗ File not found: {mesh_file}")
else:
    print("✗ mesh_discovered_paths is empty — run the discovery script first")

print()

### UMLS x MeSH